# Audiobook Studio - VoxCPM2 真引擎端到端 (Kaggle GPU T4)

目标：在 Kaggle GPU 上用**真实 VoxCPM2 引擎**完成下载模型 → 兼容补丁 → 加载 → `generate()` 合成 → 保存 .wav 全链路。
不 mock 模型/不 mock 推理。

In [ ]:
import sys, subprocess, os
print('Python', sys.version)
# 第一步：卸载所有 torch/torchaudio/transformers 等，然后从官方源重装 CUDA 12.1 版本
# 必须在任何 import 之前完成！
for pkg in ['torchaudio', 'torchvision', 'torch', 'transformers', 'accelerate', 'huggingface_hub', 'voxcpm']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg], capture_output=True)
print('old packages uninstalled')
# 从官方 PyTorch CUDA 12.1 源安装（包含 T4 CC 7.5 kernels）
# 使用 torch 2.5.1 兼容性更好
cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--index-url', 'https://download.pytorch.org/whl/cu121', '--extra-index-url', 'https://pypi.org/simple', 'torch==2.5.1', 'torchaudio==2.5.1']
print('installing torch+torchaudio CUDA12.1...')
subprocess.run(cmd, check=False)
for pkg in ['voxcpm==2.0.3','soundfile','librosa','huggingface_hub','modelscope','transformers','accelerate','safetensors']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', pkg], check=False)
print('deps ready')

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CC:', torch.cuda.get_device_capability(0))
    print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory/1e9)
    print('cuda arch:', torch.cuda.get_arch_list())

In [ ]:
import os, sys, types, torch
_real = torch.load
def _pl(*a, **k):
    k.setdefault('weights_only', False)
    return _real(*a, **k)
torch.load = _pl
sys.modules['torch'].load = _pl
if not hasattr(torch.nn, 'attention'):
    torch.nn.attention = types.ModuleType('attention')
    sys.modules['torch.nn.attention'] = torch.nn.attention
try:
    import torch.nn.attention.flex_attention as f
except Exception:
    f = types.ModuleType('flex_attention')
    torch.nn.attention.flex_attention = f
    sys.modules['torch.nn.attention.flex_attention'] = f
if not isinstance(getattr(f, 'BlockMask', None), type):
    class _BM: pass
    f.BlockMask = _BM
print('patches applied')

In [ ]:
import os
os.environ.setdefault('HF_ENDPOINT','https://hf-mirror.com')
os.environ.setdefault('HF_HUB_DISABLE_SSL_VERIFY','1')
from huggingface_hub import snapshot_download
model_dir = '/kaggle/working/VoxCPM2'
os.makedirs(model_dir, exist_ok=True)
print('Downloading openbmb/VoxCPM2 ...')
try:
    snapshot_download(repo_id='openbmb/VoxCPM2', local_dir=model_dir, resume_download=True)
except Exception as e:
    print('HF direct failed', e)
    from huggingface_hub import HfApi
    repo='openbmb/VoxCPM2'
    try:
        info = HfApi(endpoint=os.environ.get('HF_ENDPOINT','https://hf-mirror.com')).model_info(repo)
        files = sorted(s.rfilename for s in info.siblings)
        import requests
        for fn in files:
            dest=os.path.join(model_dir,fn)
            if os.path.exists(dest) and os.path.getsize(dest)>0: continue
            url=f"{os.environ.get('HF_ENDPOINT','https://hf-mirror.com')}/{repo}/resolve/main/{fn}"
            r=requests.get(url, timeout=300, stream=True, verify=False)
            if r.status_code==200:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                with open(dest,'wb') as fh:
                    for c in r.iter_content(8192): fh.write(c)
    except Exception as e2:
        raise RuntimeError(f'download failed: {e2}')
print('model files:')
for root,_,fs in os.walk(model_dir):
    for f in fs:
        print(os.path.join(root,f))

In [ ]:
import json, time
cfg_path = os.path.join(model_dir,'config.json')
cfg = json.loads(open(cfg_path).read())
if 'model_type' not in cfg:
    cfg['model_type']='voxcpm2'
    open(cfg_path,'w').write(json.dumps(cfg, indent=2))
    print('config fixed model_type=voxcpm2')
from voxcpm.modules.minicpm4.cache import StaticKVCache
def _patched_fill(self, kv_caches):
    self.current_length = kv_caches[0][0].size(2)
    self.kv_cache.zero_()
    for i in range(self.num_layers):
        key_tensor = kv_caches[i][0]
        value_tensor = kv_caches[i][1]
        if key_tensor.size(1) > self.kv_cache.size(3):
            heads_per_group = key_tensor.size(1) // self.kv_cache.size(3)
            key_tensor = key_tensor[:, ::heads_per_group, :, :]
            value_tensor = value_tensor[:, ::heads_per_group, :, :]
        self.kv_cache[0, i, :, :, : self.current_length, :] = key_tensor
        self.kv_cache[1, i, :, :, : self.current_length, :] = value_tensor
StaticKVCache.fill_caches = _patched_fill
from voxcpm import VoxCPM
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
t0=time.time()
model = VoxCPM.from_pretrained(model_dir, load_denoiser=False, optimize=False, device=dev)
sr = getattr(model.tts_model,'sample_rate', 48000)
print(f'model loaded in {time.time()-t0:.1f}s sample_rate={sr}')

In [ ]:
import numpy as np, soundfile as sf, time, os, hashlib
os.makedirs('/kaggle/working/output', exist_ok=True)
texts = [
    '你好是 VoxCPM2 模型在 Kaggle GPU 上生成的中文语音测试。',
    'This is a real VoxCPM2 text-to-speech inference test on Kaggle.',
    '三平台真引擎端到端真跟，Kaggle 节点合成音频验收。',
]
hashes={}
for i,text in enumerate(texts):
    print(f'\nTest {i+1}: {text[:40]}')
    t0=time.time()
    wav=model.generate(text=text, cfg_value=2.0, inference_timesteps=10)
    dt=time.time()-t0
    wav=np.asarray(wav).astype('float32').reshape(-1)
    out=f'/kaggle/working/output/voxcpm2_kaggle_{i}.wav'
    sf.write(out, wav, sr)
    dur=len(wav)/sr
    h=hashlib.sha256(open(out,'rb').read()).hexdigest()
    hashes[os.path.basename(out)]=h
    print(f'  OK dur={dur:.2f}s rt={dt:.2f}s bytes={os.path.getsize(out)} hash={h[:16]}')

In [ ]:
import os
print('=== output wav ===')
ok=True
for f in sorted(os.listdir('/kaggle/working/output')):
    if f.endswith('.wav'):
        p=f'/kaggle/working/output/{f}'
        import soundfile as sf
        info=sf.info(p)
        print(f'{f}: {info.duration:.2f}s {info.samplerate}Hz {os.path.getsize(p)}B')
        if info.duration<=0: ok=False
if torch.cuda.is_available():
    alloc=torch.cuda.memory_allocated()//1024//1024
    total=torch.cuda.get_device_properties(0).total_memory//1024//1024
    print(f'VRAM {alloc}/{total} MB')
print('KAGGLE_E2E_REALENGINE_PASS' if ok else 'KAGGLE_E2E_REALENGINE_FAIL')